In [ ]:
from pathlib import Path
import json
import zipfile
from IPython.display import FileLink, display

ROOT = Path.cwd()

# Handle notebooks opened one directory above the project folder.
if not (ROOT / "Assignment_PartA.ipynb").exists():
    candidate = ROOT / "LLM_Assignment_Medical_Solution"
    if (candidate / "Assignment_PartA.ipynb").exists():
        ROOT = candidate

ZIP_PATH = ROOT / "Medical_Assignment_Deliverables.zip"

part_a = ROOT / "Assignment_PartA.ipynb"
part_b = ROOT / "Assignment_PartB.ipynb"

# Part B output location used by the notebook.
instruction_dir = ROOT / "outputs" / "assignment1b_partb"

instruction_dataset = instruction_dir / "instruction_dataset.jsonl"
instruction_train = instruction_dir / "instruction_train.jsonl"
instruction_eval = instruction_dir / "instruction_eval.jsonl"

# Step 1 corpus location.
source_corpus = ROOT / "clean_corpus"
if not source_corpus.exists():
    source_corpus = ROOT / "domain_corpus"

required = [part_a, part_b, instruction_dataset, source_corpus]

missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing required deliverables: {missing}")

# Validate instruction dataset.
all_rows = [
    json.loads(line)
    for line in instruction_dataset.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

if len(all_rows) < 100:
    raise ValueError(
        f"instruction_dataset.jsonl contains only {len(all_rows)} entries; "
        "at least 100 are required."
    )

# Validate split files when available.
split_files = []
if instruction_train.exists() and instruction_eval.exists():
    train_rows = [
        json.loads(line)
        for line in instruction_train.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    eval_rows = [
        json.loads(line)
        for line in instruction_eval.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

    if len(train_rows) == 0 or len(eval_rows) == 0:
        raise ValueError("The train/eval split cannot contain an empty file.")

    split_files = [instruction_train, instruction_eval]
else:
    print("Warning: train/eval split files were not found.")

with zipfile.ZipFile(
    ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as archive:

    # Exact notebook deliverables.
    archive.write(part_a, "Assignment_PartA.ipynb")
    archive.write(part_b, "Assignment_PartB.ipynb")

    # Main instruction dataset.
    archive.write(instruction_dataset, "instruction_dataset.jsonl")

    # Supporting files documenting the train/eval split.
    for split_file in split_files:
        archive.write(split_file, split_file.name)

    # Rename clean_corpus/ to domain_corpus/ inside the archive.
    corpus_files = sorted(source_corpus.glob("*.txt"))
    if not corpus_files:
        raise ValueError(f"No .txt files found in {source_corpus}")

    for text_file in corpus_files:
        archive.write(
            text_file,
            f"domain_corpus/{text_file.name}",
        )

print(f"Created: {ZIP_PATH}")
print(f"Instruction pairs: {len(all_rows)}")
print(f"Domain text files: {len(list(source_corpus.glob('*.txt')))}")
print(f"ZIP size: {ZIP_PATH.stat().st_size / (1024 ** 2):.2f} MB")

display(FileLink(ZIP_PATH.name))